« model_18 — PV: NOKTA ve VEKTÖR · DÜZ MATEMATİK »

**Soru: Sabit noktalar (P), sabit sıra çarpanları (RM) ve öğrenilen vektör sözlüğüyle (V) kurulan PV, toplamayı öğrenebiliyor mu?**
Mimari `model_18.py` başındaki formülde. Başarı ölçütleri: success factors.

| ne | değer |
|---|---|
| veri | model_15 `veri_cok.pt` · 2+3 terim, her terim 0..500 · iz `8d0f89938c67e0e7` |
| biçim | `<eos> soru cevap <eos>` · kayıp YALNIZ cevabın rakamlarında ve EOS'ta |
| lr · yığın · adım | 0,002 · 4096 · 16.000 (model_17 MAT ile aynı) |
| ölçüm · kayıt | ölçüm ve ağırlık kaydı (`w`) her **500**, tam yedek (`t`, sürdürme) her **2.000** adımda (`MAT_COK_PV` 100 / 100 / 1.000 ile koştu); her adımın kaybı pakette (`step_losses`) |
| PV | d 128 · layer başına 256 vektör · aktif 8 · 4 layer |
| skor | sözlük sayısından (`model_18.SCORE_BY_VOCAB`): 13 token → karesiz −S_p·D, S_p 10 · `MAT_COK_PV` tabloyu ezer: −D² (kare, çarpansız) |

**Sıra:** `0 HAZIRLIK` → `1 KOŞU` ya da `1b KOŞU` → `3 NABIZ` · `4 EĞRİ` → bitince `5 SONUÇ`
**Çekirdek düşerse:** `0 HAZIRLIK` → `2 SÜRDÜR`

**Dönen hücre YOK.** Koşu arka planda bir iplikte döner, her hücre hemen geri gelir (kural 8).

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "2 SURDUR".
import os, sys, subprocess

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_18'
DATA_FILE = '/content/drive/MyDrive/model_15/veri_cok.pt'
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'],
                   check=True)
else:
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/sekerahmet/sekerai.git', REPO],
                   check=True)
SRC = REPO + '/deneme2/model_18'
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
# Yerel commit GitHub'a gitmediyse burada durur, ESKI kodla kosmaz.
assert os.path.exists(SRC + '/train.py'), 'depoda model_18 YOK -- yerelde git push gerekli'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_18', 'train', 'data'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_18, train, data

# Kural 9: veri Drive'dan; iz yuklerken YENIDEN hesaplanip karsilastirilir.
train_q, heldout_q = data.load(DATA_FILE, 'cok')
TRAIN = data.make_windows(train_q)
METRIC = data.make_metric(train_q, heldout_q, device='cuda', limit=2000)

# Kullanici, 24 Eylul: "gösterim de 500 er adım yedekler de 2000 adımda olsun"
COMMON = dict(lr=0.002, batch=4096, steps=16000, eval_every=500, save_every=2000,
             weights_every=500, seed=0)
# Skor sozluk sayisindan (SCORE_BY_VOCAB); MAT_COK_PV tablodan ONCE kosdu, kare kalir.
# Tepe (model_18) = modelin son yapisi; matematik kosulari ozellikler gelmeden onceki PV (PLAIN).
PLAIN = dict(t_max=64, start_norm=None, lam=1.0, chain='absolute', c_cache=False, cache_topk=8,
            query=False, c_content=False, select='distance', active=8, load_balance=0.0, c_m_norm=False)
CONFIGS = {'MAT_COK_PV': dict(PLAIN, d_order=128, d_content=0, vectors=256, active=8, layers=4, squared=True, S_p=1.0),
        'MAT_COK_PV_SP10': dict(PLAIN, d_order=128, d_content=0, vectors=256, active=8, layers=4)}
EXTRA = dict(data='model_15 veri_cok.pt', fingerprint=data.FINGERPRINTS['cok'], exam='MAT', code=CODE)


def memory_gb(d_order, d_content, vectors, active, layers, **score_):
    """Ileri gecisin geri yayilim icin SAKLADIGI, TAHMIN (olculmedi): layer
    basina uzaklik tablosu, aktif vektorler, C."""
    B, T, d = COMMON['batch'], TRAIN[0].shape[1], d_order + d_content      # d: D_SUM
    return layers * B * T * (vectors + active * d + 3 * d) * 4 / 1e9


n = TRAIN[0].shape[0]
print('veri  %s   iz %s   kapi GECTI' % (os.path.basename(DATA_FILE), data.FINGERPRINTS['cok']))
print('egitim %s soru   tutulan %s soru   pencere T=%d'
      % (f'{len(train_q):,}', f'{len(heldout_q):,}', TRAIN[0].shape[1]))
print('1 epok = %.0f adim   %s adim = %.0f epok'
      % (n / COMMON['batch'], f"{COMMON['steps']:,}",
         COMMON['steps'] * COMMON['batch'] / n))
for name, a in CONFIGS.items():
    _m = model_18.PV(data.N, **a)
    n_params = sum(p.numel() for p in _m.parameters())
    print('%-16s D_SUM %d  vectors %d  active %d  layers %d  skor %s   parametre %s   saklanan ~%.2f GB'
          % (name, _m.d_sum, a['vectors'], a['active'], a['layers'],
             '-%s*%s' % ('e^s' if _m.S_p_learned else '%g' % _m.S_p,
                         'D^2' if _m.squared else 'D'), f'{n_params:,}', memory_gb(**a)))

In [ ]:
# 1 KOSU BASLAT -- PV  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
RUN_1 = 'MAT_COK_PV'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[RUN_1]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train.start(RUN_1, TRAIN, data.N, metric=METRIC, device='cuda', root=ROOT,
                  extra=EXTRA, vocab=data.VOCAB, **CONFIGS[RUN_1], **COMMON))

In [ ]:
# 1b KOSU BASLAT -- PV skor tablodan (13 token: karesiz, S_p 10)  |  GPU  |  tekrar: degil -- ayni adla
#     ikinci kez calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
RUN_1B = 'MAT_COK_PV_SP10'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[RUN_1B]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train.start(RUN_1B, TRAIN, data.N, metric=METRIC, device='cuda', root=ROOT,
                  extra=EXTRA, vocab=data.VOCAB, **CONFIGS[RUN_1B], **COMMON))

In [ ]:
# 2 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.  NAME'i sec.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin STEPS'i buyut, bu hucreyi calistir.
import os, re, torch
NAME = 'MAT_COK_PV'          # CONFIGS'teki adlardan biri
STEPS = 40000              # uzatma hedefi (kural 1: surdurme)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _free, _need))

_d = ROOT + '/' + NAME
_n = sorted((int(re.findall('[0-9]+', f)[0]), f)
            for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
RESUME_FROM = _d + '/' + _n[-1][1]
print('son nokta  %s   adim %s   hedef %s' % (RESUME_FROM, f'{_n[-1][0]:,}', f'{STEPS:,}'))
assert _n[-1][0] < STEPS, 'zaten hedefe varmis -- uzatmak icin ADIM buyut'

print(train.start(NAME, TRAIN, data.N, metric=METRIC, device='cuda', root=ROOT,
                  extra=EXTRA, vocab=data.VOCAB, resume=RESUME_FROM,
                  **CONFIGS[NAME], **dict(COMMON, steps=STEPS)))

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu basar.
train.show_log(30)

In [ ]:
# 4 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Ust: SAYI (her 100 adim, sabit 2.000 + 2.000 soru).  Alt: HER ADIMIN kaybi.
import torch
import matplotlib.pyplot as plt


def log_points(name):
    """gunluk.txt -> {step: (loss, train, heldout)}."""
    r, path = {}, ROOT + '/' + name + '/gunluk.txt'
    if os.path.exists(path):
        for s in open(path, encoding='utf-8'):
            p = s.split()
            if len(p) >= 6 and p[1].isdigit():
                try:
                    r[int(p[1])] = tuple(float(x) for x in p[2:5])
                except ValueError:
                    pass
    return r


def step_losses(name):
    """Canli kosudan (train.RUNS) ya da diskteki son paketten."""
    k = train.RUNS[name].result.get('step_losses') if name in train.RUNS else None
    path = ROOT + '/model_' + name + '.pt'
    if k is None and os.path.exists(path):
        try:
            k = torch.load(path, weights_only=False,
                           map_location='cpu').get('step_losses')
        except Exception as h:          # yazilirken okunduysa
            print('  %s paketi okunamadi (%s) -- tekrar dene' % (name, h))
    return None if k is None else k[~k.isnan()]


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name in CONFIGS:
    r, k = log_points(name), step_losses(name)
    if r:
        x = sorted(r)
        a1.plot(x, [r[i][2] for i in x], label=name + ' heldout')
        a1.plot(x, [r[i][1] for i in x], ':', label=name + ' train')
        s = max(x, key=lambda i: r[i][2])
        print('%-12s %d nokta   son adim %s  heldout %.4f   en iyi %.4f (adim %s)'
              % (name, len(x), f'{x[-1]:,}', r[x[-1]][2], r[s][2], f'{s:,}'))
    if k is not None and len(k) > 200:
        a2.plot(k.numpy(), lw=0.4, label=name)
a1.axhline(0.4403, color='gray', ls='--', lw=0.8, label='model_15 tutulan 0,4403')
a1.axhline(0.2571, color='gray', ls=':', lw=0.8, label='model_17 A tutulan 0,2571')
a1.set_ylabel('accuracy (birebir dogru)')
a1.legend(fontsize=7, ncol=2)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 SONUC  |  GPU  |  tekrar: GUVENLI -- kosu BITTIKTEN sonra, sorularin TAMAMI
NAME = 'MAT_COK_PV'

# --- GPU KAPISI (CLAUDE.md kural 2)
import random
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
print('GPU kapisi GECTI: ' + torch.cuda.get_device_name(0))
assert not (NAME in train.RUNS and train.RUNS[NAME].alive), NAME + ' HALA KOSUYOR -- egitilen model olculmez'

R = train.RUNS[NAME].result if NAME in train.RUNS else {}
if 'model' not in R:                  # cekirdek yeniden basladiysa: diskteki son paket
    k = torch.load(ROOT + '/model_' + NAME + '.pt', weights_only=False,
                   map_location='cuda')
    m = model_18.PV.from_package(k).cuda()
    R = dict(k, model=m)
m = R['model'].eval()
print('%s   adim %s' % (NAME, f"{R['step']:,}"))
for measure, title in (('terim', 'KAC TERIMLI'), ('hane', 'CEVAP KAC HANELI')):
    t = data.breakdown(m, heldout_q, device='cuda', measure=measure)
    print(title)
    for a, x in t.items():
        print('%6s  heldout accuracy %.4f   first_digit %.4f   length_ok %.4f   n %d'
              % (a, x['accuracy'], x['first_digit'], x['length_ok'], x['n']))
print('\nBASAMAK (tutulan)')
data.digit_table(data.digit_check(m, heldout_q, device='cuda'))
print('\nGOZLE  (tutulandan 12 soru, tohum 7)')
data.show(m, random.Random(7).sample(heldout_q, 12), device='cuda')

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
train.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for name in CONFIGS:
    _d = ROOT + '/' + name
    if not os.path.isdir(_d):
        print('%-12s henuz kayit yok' % name)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f)
    print('%-12s %3d yedek   %.3f GB   %s'
          % (name, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi